### STEP 0: PIPELINE SETUP

In [12]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

folder = r"D:\rdkit\complete_ligand_library"

params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
pains = FilterCatalog(params)

total = 0
invalid = 0
lipinski_pass = []
pains_pass = []
final_molecules = []

for file in os.listdir(folder):

    if not file.lower().endswith(".sdf"):
        continue

    try:
        supplier = Chem.SDMolSupplier(os.path.join(folder, file))

        for mol in supplier:

            total += 1

            if mol is None:
                invalid += 1
                continue

            violations = (
                (Descriptors.MolWt(mol) > 500) +
                (Descriptors.MolLogP(mol) > 5) +
                (Lipinski.NumHDonors(mol) > 5) +
                (Lipinski.NumHAcceptors(mol) > 10)
            )

            if violations <= 1:
                lipinski_pass.append(mol)

    except OSError:
        continue

for mol in lipinski_pass:

    if not pains.HasMatch(mol):
        pains_pass.append(mol)

for mol in pains_pass:

    if Lipinski.NumRotatableBonds(mol) <= 10:
        final_molecules.append(mol)

print("\nRDKit FILTER SUMMARY")
print("=" * 40)
print("Total records:", total)
print("Invalid/unreadable:", invalid)
print("Successfully processed:", total - invalid)
print("After Lipinski:", len(lipinski_pass))
print("After PAINS:", len(pains_pass))
print("After Rotatable Bonds:", len(final_molecules))
print("=" * 40)